In [ ]:
# [목적] AI가 날짜를 일정한 형식으로 답하도록 필요한 도구와 환경을 준비합니다.
# dotenv는 API 키를 읽고, logging은 모델 실행 기록을 남깁니다.
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Chapter6-OutputParser")

In [ ]:
# [목적] AI 답변에서 날짜를 읽어 Python 날짜 객체로 바꿀 파서를 만듭니다.
# %Y-%m-%d는 2026-09-16처럼 연-월-일 순서로 날짜를 표현한다는 뜻입니다.
output_parser = DatetimeOutputParser()
output_parser.format = "%Y-%m-%d"

In [ ]:
# [목적] AI에게 전달할 날짜 출력 형식 안내를 확인합니다.
# 이 안내가 있어야 AI가 설명문 대신 정해진 날짜 형식으로 답합니다.
print(output_parser.get_format_instructions)

In [4]:
# [목적] 질문과 날짜 형식 안내를 합쳐 AI에게 보낼 프롬프트를 만듭니다.
# format_instructions는 미리 채워 두고, question만 실행할 때 바꿉니다.
template = """Answer the users question:
#Format Instructions:
{format_instructions}

#Question:
{question}

#Answer:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)

prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n#Format Instructions:\n{format_instructions}\n\n#Question:\n{question}\n\n#Answer:')

In [6]:
# [목적] 프롬프트 → AI 모델 → 날짜 파서를 하나의 실행 흐름으로 연결합니다.
# 실행 결과는 문자열이 아니라 계산에 쓸 수 있는 날짜 객체가 됩니다.
chain = prompt | ChatOpenAI() | output_parser
output = chain.invoke({"question": "Google이 창업한 연도"})

In [7]:
# [목적] 날짜 객체를 사람이 읽기 쉬운 연-월-일 문자열로 다시 표시합니다.
# strftime은 날짜를 원하는 모양의 글자로 바꿀 때 사용합니다.
output.strftime("%Y-%m-%d")

'1998-09-04'